# MapleStory Job Category Classifier, Ollama Edition
This notebook upgrades the Day 1 MapleStory classifier to use a local open-source model through Ollama.

## Workflow
MapleStory records → Exact-match retrieval → Llama 3.2 through Ollama → Output validation → Category

## Benefits

- No API costs
- Runs locally
- Uses an OpenAI-compatible interface
- Reduces hallucinations through structured retrieval
- Validates the model's output

## Install dependencies

In [ ]:
# Run this once if the packages are not installed:
# %pip install openai requests

## Import dependencies

In [1]:
import re
import requests

from openai import OpenAI

## Verify Ollama

In [ ]:
response = requests.get(
    "http://localhost:11434",
    timeout=10
)

response.raise_for_status()

print(response.text)


## Create the Ollama client

In [3]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"

client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

## Verify that Llama 3.2 is installed

In [ ]:
response = requests.get(
    "http://localhost:11434/api/tags",
    timeout=10
)

response.raise_for_status()

installed_models = [
    model["name"]
    for model in response.json().get("models", [])
]

print("Installed models:")

for model in installed_models:
    print("-", model)

model_found = any(
    model == MODEL or model.startswith(f"{MODEL}:")
    for model in installed_models
)

if not model_found:
    raise RuntimeError(
        f"{MODEL} is not installed. "
        f"Run 'ollama pull {MODEL}' in the terminal."
    )

## Quick smoke test

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Ollama is working"
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

## Create the cleaned MapleStory dataset

In [ ]:
maplestory_records = {
    "Explorers": [
        "Hero",
        "Dark Knight",
        "Paladin",
        "Bishop",
        "Arch Mage (Fire & Poison)",
        "Arch Mage (Ice & Lightning)",
        "Night Lord",
        "Shadower",
        "Bowmaster",
        "Marksman",
        "Buccaneer",
        "Corsair",
        "Dual Blade",
        "Cannoneer",
        "Pathfinder"
    ],

    "Cygnus Knights": [
        "Blaze Wizard",
        "Thunder Breaker",
        "Dawn Warrior",
        "Wind Archer",
        "Night Walker",
        "Mihile"
    ],

    "Resistance": [
        "Mechanic",
        "Battle Mage",
        "Demon Slayer",
        "Demon Avenger",
        "Wild Hunter",
        "Xenon",
        "Blaster"
    ],

    "Heroes": [
        "Mercedes",
        "Aran",
        "Phantom",
        "Luminous",
        "Evan",
        "Shade"
    ],

    "Sengoku": [
        "Kanna",
        "Hayato"
    ],

    "Nova": [
        "Angelic Buster",
        "Kaiser",
        "Cadena",
        "Kain"
    ],

    "Flora": [
        "Ark",
        "Illium",
        "Adele",
        "Khali"
    ],

    "Anima": [
        "Beast Tamer",
        "Hoyoung",
        "Lara",
        "Lynn",
        "Ren"
    ],

    "Other": [
        "Zero",
        "Kinesis",
        "Mo Xuan",
        "Sia Astelle"
    ]
}

valid_categories = list(maplestory_records.keys())

print("Number of categories:", len(valid_categories))

print(
    "Number of jobs:",
    sum(
        len(jobs)
        for jobs in maplestory_records.values()
    )
)

## Convert the data into structured records

Instead of asking Llama 3.2 to understand vague heading boundaries, create explicit records:

JOB: Kain | CATEGORY: Nova

In [ ]:
structured_records = []

for category, jobs in maplestory_records.items():
    for job in jobs:
        structured_records.append(
            f"JOB: {job} | CATEGORY: {category}"
        )

print("\n".join(structured_records[:10])) # Only print the first 10 records.

## Create the retrieval function

This retrieves only the exact job record. It prevents a small local model from becoming confused by unrelated classes.

In [8]:
def retrieve_job_record(job_name, records):
    expected_prefix = f"JOB: {job_name} |"

    matches = [
        record
        for record in records
        if record.startswith(expected_prefix)
    ]

    if not matches:
        return "NO EXACT MATCH FOUND"

    return "\n".join(matches)

Test it:

In [ ]:
print(retrieve_job_record("Kain", structured_records))
print(retrieve_job_record("Hero", structured_records))
print(retrieve_job_record("Night Lord", structured_records))

## Create the system prompt

In [11]:
system_prompt = """
You are a strict text extraction and classification system.

You will receive:

1. A requested MapleStory job name.
2. A retrieved record in this format:

JOB: <job name> | CATEGORY: <category name>

Rules:

1. Use ONLY the retrieved record.
2. The JOB value must exactly match the requested job.
3. Return the CATEGORY value from the matching record.
4. Do not use prior MapleStory knowledge.
5. Do not use semantic similarity.
6. Do not confuse the job "Hero" with the category "Heroes".
7. If the retrieved record says NO EXACT MATCH FOUND, return UNKNOWN.
8. Return only one allowed category name or UNKNOWN.
9. Do not include explanations, labels, punctuation, or extra text.
""".strip()

## Add output normalization

This removes accidental punctuation or labels.

In [12]:
def normalize_answer(answer):
    answer = answer.strip()

    answer = re.sub(
        r"^(category|answer)\s*:\s*",
        "",
        answer,
        flags=re.IGNORECASE
    )

    answer = re.sub(
        r"[^A-Za-z ]",
        "",
        answer
    )

    return answer.strip()

## Create the LLM classifier

In [13]:
def classify_job(job_name):
    retrieved_record = retrieve_job_record(
        job_name,
        structured_records
    )

    allowed_outputs = valid_categories + ["UNKNOWN"]

    user_prompt = f"""
Requested job:

{job_name}

Retrieved record:

{retrieved_record}

Allowed outputs:

{" | ".join(allowed_outputs)}

Return only the category.
""".strip()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0
    )

    raw_answer = response.choices[0].message.content
    answer = normalize_answer(raw_answer)

    if answer in allowed_outputs:
        return answer

    # Retry once if the model ignored the output instructions.
    retry_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            },
            {
                "role": "assistant",
                "content": raw_answer
            },
            {
                "role": "user",
                "content": (
                    "The previous response was invalid. "
                    "Return exactly one item from the allowed outputs "
                    "and nothing else."
                )
            }
        ],
        temperature=0
    )

    retry_answer = normalize_answer(
        retry_response.choices[0].message.content
    )

    if retry_answer in allowed_outputs:
        return retry_answer

    return "UNKNOWN"

## Test one job

In [ ]:
job_name = "Kain"

result = classify_job(job_name)

print("=" * 50)
print(job_name)
print("=" * 50)
print(result)

## Evaluate multiple jobs

In [ ]:
test_cases = {
    "Kain": "Nova",
    "Night Lord": "Explorers",
    "Angelic Buster": "Nova",
    "Cadena": "Nova",
    "Hero": "Explorers",
    "Mercedes": "Heroes",
    "Night Walker": "Cygnus Knights",
    "Adele": "Flora",
    "Lara": "Anima",
    "Not A MapleStory Job": "UNKNOWN"
}

passed_tests = 0

for job, expected_category in test_cases.items():
    predicted_category = classify_job(job)

    passed = predicted_category == expected_category

    if passed:
        passed_tests += 1

    print("\n" + "=" * 60)
    print("Job:", job)
    print("Expected:", expected_category)
    print("Predicted:", predicted_category)
    print("Result:", "PASS" if passed else "FAIL")

## Calculate accuracy

In [ ]:
total_tests = len(test_cases)
accuracy = passed_tests / total_tests

print("\n" + "=" * 60)
print("Final Evaluation")
print("=" * 60)

print(f"Passed: {passed_tests}/{total_tests}")
print(f"Accuracy: {accuracy:.0%}")

## Reflection
This experiment showed that Llama 3.2 struggled when given large, noisy webpage content.

The improved design uses structured data and retrieves only the relevant job record before sending it to the LLM. This reduced ambiguity while keeping the LLM responsible for the final classification.


**Key Learning**

Reliable LLM applications depend on more than prompt engineering. Clean context, retrieval, validation, and testing can significantly improve model reliability.